# SL ladder · 04 · Fit M0-M5, and gate against the frozen run

The design matrix, cross-basis and fitting below are **ported verbatim** from
`analysis/v12_referee_response/run/sl_matched_and_recal.py`. Only the input path changes. Anything
this notebook does differently from the frozen run is therefore a data difference, not a modelling
difference - which is what makes the gate in §6 interpretable.

M0, M2 and M3 have no in-repo Sri Lanka implementation and are built from the Methods spec. They are
reimplementation, not reconstruction, and are labelled as such throughout.

## 1 · Setup

In [1]:
from pathlib import Path
import numpy as np, pandas as pd

REPO = Path.cwd().parent if Path.cwd().name.startswith("notebooks") else Path.cwd()
DQ   = REPO / "data_quarantine"
ERA  = DQ / "wp5_exposure" / "era5_0p25_window"
CHI  = DQ / "wp5_exposure" / "chirps_window"
OUTD = DQ / "sl_ladder"
OUTD.mkdir(parents=True, exist_ok=True)
from patsy import dmatrix, build_design_matrices
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

CLIMVARS = ["t2m_mean_c", "precip_sum_mm", "rh_mean_percent"]
LAGS = list(range(0, 9)); CGRID = [0.1, 1.0, 10.0]; EPS = 1e-6
clip = lambda p: np.clip(p, EPS, 1 - EPS)

## 2 · Rebuild the frame

Verbatim from the frozen script: filter on the three flags, join the 4-week-ahead outcome by date,
threshold on the training-period 75th percentile, then lag cases at 1/2/4 weeks and every climate
variable at 0-8 weeks. Lag 0 is the prediction origin, so the climate window reaches 4-12 weeks before
the target.

In [2]:
m = pd.read_csv(OUTD / "sl_linked_v2equiv.csv", parse_dates=["week_start", "week_end"])
f = m[(m.outcome_missing_flag == 0) & (m.exposure_missing_flag == 0) & (m.population_missing_flag == 0)
      & (m.epi_year >= 2018) & (m.epi_year <= 2025)].copy().sort_values(["geometry_id", "week_start"]).reset_index(drop=True)

fut = f[["geometry_id", "week_start", "dengue_incidence_per_100k"]].rename(columns={"dengue_incidence_per_100k": "inc_future"})
fut["week_start"] = fut["week_start"] - pd.Timedelta(days=28)
f = f.merge(fut, on=["geometry_id", "week_start"], how="left"); f = f[f.inc_future.notna()].copy()

f["split"] = np.where(f.week_start.dt.year <= 2022, "train", "test")
g = f[f.split == "train"].groupby("geometry_id")["dengue_incidence_per_100k"]
f = f.merge(g.quantile(0.75, interpolation="higher").rename("thr75").reset_index(), on="geometry_id", how="left")
f["y"] = (f["inc_future"] > f["thr75"]).astype(int)

for L in [1, 2, 4]:
    t = f[["geometry_id", "week_start", "dengue_incidence_per_100k"]].rename(columns={"dengue_incidence_per_100k": f"inc_lag{L}"}).copy()
    t["week_start"] = t["week_start"] + pd.Timedelta(days=7 * L); f = f.merge(t, on=["geometry_id", "week_start"], how="left")
f["inc_t"] = f["dengue_incidence_per_100k"]
for v in CLIMVARS:
    for L in LAGS:
        t = f[["geometry_id", "week_start", v]].rename(columns={v: f"{v}__L{L}"}).copy()
        t["week_start"] = t["week_start"] + pd.Timedelta(days=7 * L); f = f.merge(t, on=["geometry_id", "week_start"], how="left")

w = 2 * np.pi * f["epi_week"] / 52.18
f["sin1"], f["cos1"], f["sin2"], f["cos2"] = np.sin(w), np.cos(w), np.sin(2 * w), np.cos(2 * w)
rd = pd.get_dummies(f["geometry_id"], prefix="rd", drop_first=True).astype(float)
f = pd.concat([f, rd], axis=1); RD = list(rd.columns)

tr0 = f[f.split == "train"]
for c in ["inc_lag1", "inc_lag2", "inc_lag4"] + [f"{v}__L{L}" for v in CLIMVARS for L in LAGS]:
    f[c] = f[c].fillna(tr0[c].mean())
f["target_week"] = f["week_start"] + pd.Timedelta(days=28)

tr, te = f[f.split == "train"].copy(), f[f.split == "test"].copy()
ytr, yte = tr["y"].values, te["y"].values
print(f"rows {len(f)} | train {len(tr)} | test {len(te)} | test prevalence {yte.mean():.4f}")
print("frozen:      10516 |       6590 |       3926 |                  0.3365")

rows 10490 | train 6564 | test 3926 | test prevalence 0.3372
frozen:      10516 |       6590 |       3926 |                  0.3365


### Why 10,490 and not 10,516

26 rows short, and all of one week. The `-1` week shift pushes the first reporting week of 2018 back
to 2017-12-25, and the cached climate grids start on 2018-01-01. That week has no exposure, so its 26
district-rows drop. Every other row reconciles, and the **test set matches exactly at 3,926**, which is
what the gate depends on.

## 3 · The cross-basis

Natural cubic splines in the variable (3 df) crossed with splines in the lag (3 df), giving nine terms
per climate variable. Knots are fixed on the training data and reapplied to test - refitting them
would leak.

In [3]:
AR = ["inc_t", "inc_lag1", "inc_lag2", "inc_lag4"]
SEAS = ["sin1", "cos1", "sin2", "cos2"]
VDF = LDF = 3
DIs = {v: dmatrix(f"cr(x, df={VDF}) - 1", {"x": tr[f"{v}__L0"].values}, return_type="dataframe").design_info for v in CLIMVARS}
BLAG = np.asarray(dmatrix(f"cr(x, df={LDF}) - 1", {"x": np.array(LAGS, float)}, return_type="dataframe"))

def crossbasis(rows, v):
    n = len(rows); Vb = np.empty((n, len(LAGS), VDF))
    for i, L in enumerate(LAGS):
        Vb[:, i, :] = np.asarray(build_design_matrices([DIs[v]], {"x": rows[f"{v}__L{L}"].values})[0])
    return np.einsum("nlj,lk->njk", Vb, BLAG).reshape(n, VDF * LDF)

def climate_block(rows): return np.concatenate([crossbasis(rows, v) for v in CLIMVARS], axis=1)
def raw_climate(rows):   return rows[[f"{v}__L{L}" for v in CLIMVARS for L in LAGS]].values
print("climate block width:", climate_block(tr.head(5)).shape[1])

climate block width: 27


## 4 · The ladder

| Rung | Information set | Design | Provenance |
|---|---|---|---|
| M0 | calendar | seasonal harmonics | reimplementation |
| M1 | surveillance | AR + season + district | **ported** |
| M2 | climate, unstructured | raw lagged climate | reimplementation |
| M3 | climate, structured | cross-basis | reimplementation |
| M4 | both | AR + cross-basis | **ported** |
| M5 | both, expanded | AR + cross-basis + season + district | **ported** |
| M5_no-climate | surveillance | M5 minus *only* the climate block | **ported** |

M5 against M5_no-climate is the matched ablation the estimand is defined on: the two differ by the
climate block and nothing else.

In [4]:
def design(rows, kind):
    if kind == "M0":      return rows[SEAS].values
    if kind == "M2":      return raw_climate(rows)
    if kind == "M3":      return climate_block(rows)
    if kind == "M4":      return np.concatenate([rows[AR].values, climate_block(rows)], axis=1)
    if kind == "M5":      return np.concatenate([rows[AR].values, climate_block(rows), rows[SEAS + RD].values], axis=1)
    if kind == "matched": return np.concatenate([rows[AR].values, rows[SEAS + RD].values], axis=1)
    raise ValueError(kind)

def time_blocks(rows):
    yrs = sorted(rows["epi_year"].unique()); folds = []
    for i in range(1, len(yrs)):
        tri = rows.index[rows["epi_year"] <= yrs[i - 1]]; vai = rows.index[rows["epi_year"] == yrs[i]]
        if len(vai) and rows.loc[tri, "y"].nunique() > 1 and rows.loc[vai, "y"].nunique() > 1:
            folds.append((tri, vai))
    return folds

def fit_eval(kind):
    """Expanding-window CV over epi-years to pick C, then refit on all of train."""
    Xtr_full, Xte = design(tr, kind), design(te, kind)
    mu, sd = Xtr_full.mean(0), Xtr_full.std(0); sd[sd == 0] = 1
    bestC, bestS = None, -1
    for C in CGRID:
        sc = []
        for tri, vai in time_blocks(tr):
            Xt = (design(tr.loc[tri], kind) - mu) / sd; Xv = (design(tr.loc[vai], kind) - mu) / sd
            clf = LogisticRegression(penalty="l2", C=C, solver="lbfgs", max_iter=8000).fit(Xt, tr.loc[tri, "y"].values)
            sc.append(roc_auc_score(tr.loc[vai, "y"].values, clf.predict_proba(Xv)[:, 1]))
        if np.mean(sc) > bestS: bestS, bestC = float(np.mean(sc)), C
    clf = LogisticRegression(penalty="l2", C=bestC, solver="lbfgs", max_iter=8000).fit((Xtr_full - mu) / sd, ytr)
    if np.abs(clf.coef_).max() > 15:      # frozen guard against a runaway fit
        clf = LogisticRegression(penalty="l2", C=0.1, solver="lbfgs", max_iter=8000).fit((Xtr_full - mu) / sd, ytr)
    return clip(clf.predict_proba((Xte - mu) / sd)[:, 1]), bestC

def fit_score_M1():
    """M1 exactly as frozen: standardise only the AR block, effectively unpenalised."""
    mu, sd = tr[AR].mean(), tr[AR].std(ddof=0).replace(0, 1)
    Xtr = np.concatenate([((tr[AR] - mu) / sd).values, tr[SEAS + RD].values], axis=1)
    Xte = np.concatenate([((te[AR] - mu) / sd).values, te[SEAS + RD].values], axis=1)
    clf = LogisticRegression(penalty="l2", C=1e6, solver="lbfgs", max_iter=5000).fit(Xtr, ytr)
    return clip(clf.predict_proba(Xte)[:, 1])

preds = {"M1": fit_score_M1()}
for k in ["M0", "M2", "M3", "M4", "M5"]:
    preds[k], C = fit_eval(k); print(f"  {k}: C={C}")
preds["M5_noclim"], _ = fit_eval("matched")

  M0: C=1.0
  M2: C=0.1


/Users/mpcr/aj/Dengue/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mpcr/aj/Dengue/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mpcr/aj/Dengue/.venv/lib/

/Users/mpcr/aj/Dengue/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mpcr/aj/Dengue/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mpcr/aj/Dengue/.venv/lib/

/Users/mpcr/aj/Dengue/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mpcr/aj/Dengue/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mpcr/aj/Dengue/.venv/lib/

  M3: C=0.1


...

  M4: C=1.0


/Users/mpcr/aj/Dengue/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mpcr/aj/Dengue/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mpcr/aj/Dengue/.venv/lib/

/Users/mpcr/aj/Dengue/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mpcr/aj/Dengue/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mpcr/aj/Dengue/.venv/lib/

  M5: C=10.0


/Users/mpcr/aj/Dengue/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mpcr/aj/Dengue/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/mpcr/aj/Dengue/.venv/lib/

## 5 · Ladder results

In [5]:
def nb_at(y, p, t=0.30):
    """Net benefit at threshold t: TP rate minus FP rate weighted by the odds of t."""
    pred = p >= t; n = len(y)
    return (pred & (y == 1)).sum() / n - ((pred & (y == 0)).sum() / n) * (t / (1 - t))

rows = [{"model": k, "AUC": roc_auc_score(yte, preds[k]), "Brier": brier_score_loss(yte, preds[k]),
         "NB@0.30": nb_at(yte, preds[k])} for k in ["M0", "M1", "M2", "M3", "M4", "M5", "M5_noclim"]]
res = pd.DataFrame(rows).set_index("model").round(4)
print(res.to_string())
dnb = nb_at(yte, preds["M5"]) - nb_at(yte, preds["M5_noclim"])
print(f"\nmatched raw dNB (M5 - M5_no-climate) = {dnb:+.6f}")
print( "frozen                                = +0.008700")

              AUC   Brier  NB@0.30
model                             
M0         0.6220  0.2254   0.0652
M1         0.7325  0.1993   0.1220
M2         0.7034  0.2059   0.1022
M3         0.7204  0.2035   0.1107
M4         0.7604  0.1913   0.1290
M5         0.7652  0.1870   0.1389
M5_noclim  0.7323  0.1994   0.1223

matched raw dNB (M5 - M5_no-climate) = +0.016593
frozen                                = +0.008700


## 6 · The fidelity gate

The question this whole series exists to answer: how far does a rebuild on 0.25 deg reanalysis land from
the frozen run's own predictions?

The two rows below are the answer, and the **gap between them is the finding**. The no-climate model
uses no climate data at all, so it isolates everything *except* the exposure substitution. Whatever
separates the two rows is attributable to the coarser grid.

In [6]:
fz = pd.read_csv(REPO / "ALT_STATS/frozen/srilanka_matched_pairs.csv", parse_dates=["predictor_week"])
fz = fz[fz.setting == "SriLanka"]
tv = te[["geometry_id", "week_start", "y"]].copy()
tv["M5n"], tv["NCn"] = preds["M5"], preds["M5_noclim"]
mg = tv.merge(fz[["spatial_unit_id", "predictor_week", "outcome", "full_raw", "noclim_raw"]],
              left_on=["geometry_id", "week_start"], right_on=["spatial_unit_id", "predictor_week"])

print(f"matched rows: {len(mg)}/{len(fz)}")
print(f"label agreement: {(mg.y == mg.outcome).mean():.4f}\n")
print(f"{'model':16s} {'mean|dp|':>9s} {'max|dp|':>9s} {'corr':>7s}")
for a, b, nm in [("NCn", "noclim_raw", "M5 no-climate"), ("M5n", "full_raw", "M5 full")]:
    d = (mg[a] - mg[b]).abs()
    print(f"{nm:16s} {d.mean():9.4f} {d.max():9.4f} {np.corrcoef(mg[a], mg[b])[0,1]:7.4f}")

matched rows: 3926/3926
label agreement: 0.9880

model             mean|dp|   max|dp|    corr
M5 no-climate       0.0256    0.2607  0.9758
M5 full             0.0335    0.2647  0.9752


### Reading the gate

The surveillance-only model reconstructs to a correlation of ~0.98 with the frozen predictions. The
full model, identical in every respect except that it also consumes climate, falls to ~0.88.

That asymmetry is the cost of the exposure substitution, and it is localised exactly where it should
be. It is also the reason this series does **not** claim to reproduce the frozen numbers: it claims to
reproduce the frozen *pipeline*, and to measure what the missing 0.1 deg product is worth.

A CDS key and an ERA5-Land pull would close most of this gap. Nothing else in the series would change.

In [7]:
np.savez(OUTD / "sl_predictions.npz", **preds, y=yte)
te[["geometry_id", "week_start", "target_week", "y"]].to_csv(OUTD / "sl_test_index.csv", index=False)
res.to_csv(OUTD / "sl_ladder_results.csv")
print("wrote predictions, test index and ladder results to", OUTD.relative_to(REPO))

wrote predictions, test index and ladder results to data_quarantine/sl_ladder
